# State Hydro and Flood Data Extraction

This notebook runs reusable extractors that download, standardize, and cache selected hydrographic, flood-hazard, exposure, land-cover, and geographic reference data for one U.S. state.

Change the configuration values in the **Setup** section to select a different state, date range, or land-cover year. Each extractor saves a local output in `data/raw/` in GeoParquet or GeoTiff format and may reuse that output on later runs. Most extractors skip an existing output, so delete a specific output file when a refresh is needed. Raw data should remain excluded from Git because they are downloaded outputs and/or local credentials.

## Included datasets

1. USGS stream gauges with site attributes and available record information.
2. FEMA National Flood Hazard Layer (NFHL) effective flood-hazard polygons.
3. USACE National Structure Inventory (NSI) structure points.
4. Census TIGER/Line county boundaries.
5. Census TIGER/Line county-subdivision boundaries.
6. Census tract boundaries with 2020 urban and rural housing-unit counts.
7. Annual National Land Cover Database (NLCD) land-cover raster.
8. NOAA Storm Events records for flood-related events.
9. OpenFEMA NFIP redacted claims records.
10. USGS Watershed Boundary Dataset HUC12 subwatershed polygons.
11. USDA Rural-Urban Continuum Codes.

## Setup

Run this cell before any extractor.

The configuration block defines:

- `STATE_FIPS`: Two-digit Census state FIPS code.
- `STATE_ABBR`: Two-letter postal abbreviation used in filenames.
- `STATE_NAME`: Readable state name used in notebook output.
- `YEAR_START` and `YEAR_END`: Inclusive date range for sources that support time filtering.
- `LAND_COVER_YEAR`: Annual NLCD year to download and clip.

`RAW_DIR` is the local cache directory. Most extractors return a previously saved result unless their `force=True` option is used.

In [1]:
import sys
import os
import io
import re
import time
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rasterio
import requests
from shapely.geometry import Point, LineString

from dotenv import load_dotenv
load_dotenv()

sys.path.append("../src")

# Below is for .py files only
# PROJECT_ROOT = Path(__file__).resolve().parent.parent
# RAW_DIR = PROJECT_ROOT / "data" / "raw"
# RAW_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

# ---------------------------------------------------------------------------
# Config — change lines below to run for a different state or time period
# ---------------------------------------------------------------------------
STATE_FIPS = "44"  # Rhode Island
STATE_ABBR = "RI"
STATE_NAME = "Rhode Island"

# STATE_FIPS = "25"  # Massachusetts
# STATE_ABBR = "MA"
# STATE_NAME = "Massachusetts"

YEAR_START = 2010
YEAR_END = 2024
LAND_COVER_YEAR = 2024

print(f"State: {STATE_NAME} ({STATE_ABBR})")
print(f"Year range: {YEAR_START}-{YEAR_END}")
print(f"Raw data dir: {RAW_DIR.resolve()}")

State: Rhode Island (RI)
Year range: 2010-2024
Raw data dir: /Users/pamelagreen/Desktop/Career/Python/state_hydro_extraction/data/raw


## USGS Stream Gauges

This section retrieves USGS National Water Information System site records for the selected state. The output includes stream-gauge locations and available site attributes used to identify monitoring stations with hydrologic observations.

**Output:** `data/raw/usgs_gauges_<STATE>.parquet`

Review the returned GeoDataFrame to confirm the number of sites and inspect fields such as site identifier, station name, location, and available record information.

In [2]:
from usgs_stream_gauges import extract_usgs_stream_gauges

gauges_gdf = extract_usgs_stream_gauges(
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
    force=False,
)

gauges_gdf.head()

/Users/pamelagreen/miniforge3/envs/mgis/lib/python3.14/site-packages/dataretrieval/nwis.py:103: DataCurrencyWarning: Starting in March 2024, the NWIS qw data endpoint is retiring and no longer receives updates. For more information, refer to https://waterdata.usgs.gov/nwis/qwdata and https://doi-usgs.github.io/dataRetrieval/articles/Status.html or email CompTools@usgs.gov.
  return func(*args, **kwargs)


,site_no,station_nm,dec_lat_va,dec_long_va,site_tp_cd,huc_cd,begin_date,end_date,is_active_now,geometry
1,01106000,"ADAMSVILLE BROOK AT ADAMSVILLE, RI",41.558314,-71.129192,ST,1.090002e+10,1940-10-01,2026-09-06,True,POINT (-71.12919 41.55831)
14,01109403,"TEN MILE R., PAWTUCKET AVE. AT E. PROVIDENCE, RI",41.830934,-71.350331,ST,1.090004e+06,1970-09-30,2026-09-06,True,POINT (-71.35033 41.83093)
26,01111270,"CLEAR RIVER AT HARRISVILLE, RI",41.970098,-71.684789,ST,1.090003e+06,1968-09-04,1968-09-04,False,POINT (-71.68479 41.9701)
38,01111300,"NIPMUC RIVER NEAR HARRISVILLE, RI",41.981209,-71.685900,ST,1.090003e+06,1964-03-01,2026-09-06,True,POINT (-71.6859 41.98121)
48,01111390,"SPRING GROVE POND OUTLET AT CHEPACHET, RI",41.922599,-71.657288,ST,1.090003e+06,1968-09-04,1968-09-04,False,POINT (-71.65729 41.9226)


## FEMA Flood-Hazard Polygons

This section downloads FEMA’s effective state-level National Flood Hazard Layer package from the FEMA Map Service Center and extracts the `SFLDHAZAR` flood-hazard-area layer.

The result contains mapped flood-hazard polygons and FEMA attributes such as `FLDZONE`, `ZONESUBTY`, and `SFHATF`. It represents FEMA’s effective flood-hazard mapping, not a prediction of current flood conditions.

**Output:** `data/raw/nfhl_flood_zones_<STATE>_MSC.parquet`

In [3]:
from extract_nfhl_msc import extract_nfhl_flood_zones_msc

nfhl_gdf = extract_nfhl_flood_zones_msc(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
)

 Getting a valid county FIPS for RI...
 Using county FIPS: 44009
 Looking up a FEMA community ID for county 44009...
 Using community ID: 445395
 Looking up the current statewide NFHL package for RI...
 Current state product: NFHL_44_20260622 (effective 06/23/2026, posted None, size 52MB)
 Downloaded 53.0 MB. Extracting...
 Found geodatabase: NFHL_44_20260622.gdb
 Loading layer S_FLD_HAZ_AR...


/Users/pamelagreen/miniforge3/envs/mgis/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


 Loaded 8,347 flood-hazard polygons (CRS: EPSG:4269).
 Saved 8,347 NFHL flood-zone polygons to /Users/pamelagreen/Desktop/Career/Python/state_hydro_extraction/data/raw/nfhl_flood_zones_RI_MSC.parquet


## USACE National Structure Inventory

This section retrieves point-level structure records from the USACE National Structure Inventory API. The extractor queries the selected state by county and returns a GeoDataFrame of available structures.

Typical fields may include structure and content values, building area, foundation characteristics, ground elevation, and an associated FEMA flood-zone value. Field availability can vary by location and data release.

**Output:** `data/raw/nsi_structures_<STATE>.parquet`

In [4]:
from usace_nsi_structures import extract_nsi_structures

nsi_gdf = extract_nsi_structures(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
)

Querying NSI: Bristol (44001)
Querying NSI: Kent (44003)
Querying NSI: Newport (44005)
Querying NSI: Providence (44007)
Querying NSI: Washington (44009)
firmzone null rate: 0.0%


## Census County Boundaries

This section downloads Census TIGER/Line county polygons and filters the nationwide source to the selected state. County boundaries provide a stable reference geography for state-level extraction and mapping.

The saved GeoParquet retains Census geographic identifiers and attributes along with geometries standardized to EPSG:4326.

**Output:** `data/raw/counties_<STATE>.parquet`

In [5]:
from census_counties import extract_county_boundaries

counties_gdf = extract_county_boundaries(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
)

print(f"{len(counties_gdf):,} county boundaries")
counties_gdf.head()

5 county boundaries


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,44,009,01219782,44009,0500000US44009,Washington,Washington County,06,H4,G4020,148,39300,NaN,N,852827596,604769732,+41.3967920,-071.6202820,"MULTIPOLYGON (((-71.5752 41.32094, -71.57525 4..."
1,44,007,01219781,44007,0500000US44007,Providence,Providence County,06,H4,G4020,148,39300,NaN,N,1060552451,67870210,+41.8697678,-071.5786246,"POLYGON ((-71.54735 41.7312, -71.54743 41.7312..."
2,44,001,01219777,44001,0500000US44001,Bristol,Bristol County,06,H4,G4020,148,39300,NaN,N,62500773,53359134,+41.7068397,-071.2866874,"POLYGON ((-71.21043 41.68801, -71.21086 41.687..."
3,44,005,01219779,44005,0500000US44005,Newport,Newport County,06,H4,G4020,148,39300,NaN,N,265293779,547001789,+41.5010449,-071.2830626,"POLYGON ((-71.1164 41.48457, -71.11543 41.4827..."
4,44,003,01219778,44003,0500000US44003,Kent,Kent County,06,H4,G4020,148,39300,NaN,N,436588773,50686111,+41.6751180,-071.5802819,"POLYGON ((-71.54481 41.60199, -71.54501 41.601..."


## Census County Subdivisions

This section downloads TIGER/Line county-subdivision polygons for the selected state. County subdivisions are Census geographic units that may represent towns, townships, boroughs, precinct-like entities, or other subcounty areas, depending on the state.

Interpret the layer using the Census definition appropriate to the selected state; county subdivisions do not have identical governmental meaning nationwide.

**Output:** `data/raw/county_subdivisions_<STATE>.parquet`

In [6]:
from census_county_subdivisions import extract_county_subdivisions

county_subdivisions_gdf = extract_county_subdivisions(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
    tiger_year=2023,
)

print(f"{len(county_subdivisions_gdf):,} county subdivisions")
county_subdivisions_gdf.head()

40 county subdivisions


,STATEFP,COUNTYFP,COUSUBFP,COUSUBNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,44,009,14500,01220080,4400914500,0600000US4400914500,Charlestown,Charlestown town,43,T1,G4040,A,94641259,59366507,+41.3713678,-071.6803372,"POLYGON ((-71.73125 41.39851, -71.73109 41.398..."
1,44,005,70880,01220066,4400570880,0600000US4400570880,Tiverton,Tiverton town,43,T1,G4040,A,75254509,18875638,+41.6069833,-071.1826339,"POLYGON ((-71.22475 41.56324, -71.22426 41.570..."
2,44,005,45460,01220063,4400545460,0600000US4400545460,Middletown,Middletown town,43,T1,G4040,A,32891674,5406460,+41.5201518,-071.2801985,"POLYGON ((-71.32328 41.53566, -71.32238 41.538..."
3,44,005,00000,00000000,4400500000,0600000US4400500000,County subdivisions not defined,County subdivisions not defined,00,Z9,G4040,F,0,330539964,+41.4102788,-071.3027192,"POLYGON ((-71.50944 41.30773, -71.5066 41.3126..."
4,44,009,77000,01220091,4400977000,0600000US4400977000,Westerly,Westerly town,43,T1,G4040,A,76323893,116618435,+41.3234277,-071.8024584,"POLYGON ((-71.90722 41.30454, -71.90711 41.304..."


## Census Tracts and Urban/Rural Housing Units

This section combines Census TIGER/Line tract polygons with 2020 Decennial Census Demographic and Housing Characteristics table H2.

The output includes total housing units, housing units in urban areas, housing units in rural areas, and the calculated shares `pct_urban` and `pct_rural`. The extractor needs a Census API key, supplied through `CENSUS_API_KEY` in a local `.env` file or passed as a function argument.

**Output:** `data/raw/census_tract_urban_rural_<STATE>.parquet`

In [7]:
from census_tract_urban_rural import extract_tract_urban_rural

tract_urban_rural_gdf = extract_tract_urban_rural(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
)

print(f"{len(tract_urban_rural_gdf):,} Census tracts")
tract_urban_rural_gdf[
    [
        "geoid",
        "county_fips",
        "total_units",
        "urban_units",
        "rural_units",
        "pct_urban",
        "pct_rural",
    ]
].head()

250 Census tracts


,geoid,county_fips,total_units,urban_units,rural_units,pct_urban,pct_rural
0,44005040105,44005,1410,872,538,0.61844,0.38156
1,44005040104,44005,2256,2256,0,1.0,0.0
2,44009050702,44009,1764,608,1156,0.344671,0.655329
3,44009050701,44009,1849,58,1791,0.031368,0.968632
4,44007000302,44007,1816,1816,0,1.0,0.0


## NOAA Flood-Related Storm Events

This section retrieves NOAA Storm Events Database records for the selected state and configured year range. The extractor retains flood-related event types and standardizes event dates, locations, damages, injuries, fatalities, and county or zone references when available.

Storm Events records describe reported weather events and their documented impacts. Geographic precision and field completeness can vary by event, location type, and reporting year.

**Output:** `data/raw/noaa_storm_events_<STATE>.parquet`

In [8]:
from noaa_storm_events import extract_noaa_storm_events

storm_events_gdf = extract_noaa_storm_events(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    state_name=STATE_NAME,
    year_start=YEAR_START,
    year_end=YEAR_END,
    output_dir=RAW_DIR,
)

## OpenFEMA NFIP Redacted Claims

This section retrieves National Flood Insurance Program claims data from the OpenFEMA API and filters records to the selected state.

The public dataset is redacted to protect privacy. It should be interpreted as a record of reported and processed NFIP claims, not a complete inventory of all flood losses or all structures exposed to flooding.

**Output:** `data/raw/nfip_claims_<STATE>.parquet`

In [9]:
from openfema_nfip_claims import extract_nfip_claims

claims_df = extract_nfip_claims(
    state_abbr=STATE_ABBR,
    year_start=YEAR_START,
    year_end=YEAR_END,
    output_dir=RAW_DIR,
)

Fetching NFIP claims: offset 0
Fetching NFIP claims: offset 1,000
Fetching NFIP claims: offset 2,000
Fetching NFIP claims: offset 3,000


## USGS HUC12 Subwatersheds

This section downloads Watershed Boundary Dataset polygons and filters them to HUC12-scale subwatersheds that intersect the selected state.

A HUC12 is a standardized hydrologic unit defined by drainage geography rather than administrative boundaries. Subwatersheds may cross state borders, so an intersecting HUC12 layer can extend beyond the selected state unless it is explicitly clipped.

**Output:** `data/raw/huc12_subwatersheds_<STATE>.parquet`

In [10]:
from usgs_wbd_huc12_subwatersheds import extract_wbd_huc12_subwatersheds

huc12_gdf = extract_wbd_huc12_subwatersheds(
    state_abbr=STATE_ABBR,
    output_dir=RAW_DIR,
)

Requesting HUC12 features starting at record 0...
Saved 61 full HUC12 subwatershed polygons to /Users/pamelagreen/Desktop/Career/Python/state_hydro_extraction/data/raw/usgs_wbd_huc12_subwatersheds_RI.parquet.


## Annual NLCD Land Cover

This section downloads the full-CONUS Annual National Land Cover Database archive for the requested year and clips the raster to the selected state boundary.

Annual NLCD is a 30-meter categorical land-cover product. The clipped output retains NLCD class values inside the state and uses NoData outside the state boundary. The initial full-CONUS download is large; use `keep_source=True` only when you want to retain the original ZIP and GeoTIFF for clipping additional states.

**Output:** `data/raw/mrlc_land_cover_<STATE>_<YEAR>.tif`

In [11]:
from mrlc_land_cover import extract_mrlc_land_cover

land_cover_path = extract_mrlc_land_cover(
    state_fips=STATE_FIPS,
    state_abbr=STATE_ABBR,
    year=LAND_COVER_YEAR,
    output_dir=RAW_DIR,
    keep_source=False,
)

Extracting the Annual NLCD GeoTIFF...
Saved clipped 2024 land-cover raster: /Users/pamelagreen/Desktop/Career/Python/state_hydro_extraction/data/raw/mrlc_land_cover_RI_2024.tif
Deleted cached full-CONUS NLCD source files.


## Review Downloaded Outputs

Run the following cell after completing the desired extractors to list files currently stored in `data/raw/`.

Check that expected outputs exist, have plausible file sizes, and use the selected state abbreviation in their names. Re-run an individual extractor with `force=True` only when you deliberately want to replace its cached result.

In [12]:
from pathlib import Path

if not RAW_DIR.exists():
    print(f"Raw-data directory does not exist: {RAW_DIR}")
else:
    files = sorted(path for path in RAW_DIR.iterdir() if path.is_file())

    if not files:
        print(f"No extracted files found in: {RAW_DIR}")
    else:
        print(f"Files in: {RAW_DIR}\n")

        for path in files:
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{path.name:50s} {size_mb:8.2f} MB")

Files in: /Users/pamelagreen/Desktop/Career/Python/state_hydro_extraction/data/raw

.DS_Store                                              0.01 MB
census_tract_urban_rural_RI.parquet                    1.30 MB
counties_RI.parquet                                    0.12 MB
county_subdivisions_RI.parquet                         0.60 MB
mrlc_land_cover_RI_2024.tif                            8.13 MB
nfhl_flood_zones_RI_MSC.parquet                       79.31 MB
nfip_claims_RI_2010_2024.parquet                       0.40 MB
noaa_storm_events_RI.parquet                           0.11 MB
noaa_zone_county_correlation.parquet                   0.18 MB
nsi_structures_RI.parquet                             43.19 MB
usgs_stream_gauges_RI.parquet                          0.02 MB
usgs_wbd_huc12_subwatersheds_RI.parquet                1.81 MB
